# BERTimbau Large

Training of BERTimbau Large across the three corpus organizations:

1. `data/original/`
2. `data/pair_controlled/seed_*`
3. `data/max_cross_split/seed_*`

The model and its hyperparameters remain fixed. The training seed is always 40.

For each trained model, TP, TN, FP, FN, the confusion matrix, metrics, training history, metadata, and the analysis of pun-related signals observed in false negatives are saved.

In [1]:
from pathlib import Path
import gc
import inspect
import json
import platform
import random
import re
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers

from torch.utils.data import Dataset

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

from transformers.utils import logging as hf_logging

from sklearn import __version__ as sklearn_version
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

In [ ]:
MODEL_NAME = "neuralmind/bert-large-portuguese-cased"

MODEL_SEED = 40

SPLIT_SEEDS = [
    13,
    21,
    40,
    42,
    73,
    101,
]

STRATEGIES = [
    "pair_controlled",
    "max_cross_split",
]

SPLIT_NAMES = [
    "train",
    "validation",
    "test",
]

EXPECTED_SPLIT_COUNTS = {
    "train": 3990,
    "validation": 570,
    "test": 1140,
}

EXPECTED_CLASS_COUNTS = {
    "train": {0: 1995, 1: 1995},
    "validation": {0: 285, 1: 285},
    "test": {0: 570, 1: 570},
}

ERROR_CATEGORIES = [
    "none",
    "homophone_only",
    "homograph_only",
    "both",
]

MAX_LENGTH = 256
NUM_EPOCHS = 6
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 2

ID2LABEL = {
    0: "0",
    1: "1",
}

LABEL2ID = {
    "0": 0,
    "1": 1,
}

In [3]:
def find_project_root(start_path=None):
    current = Path(start_path or Path.cwd()).resolve()

    while True:
        if (current / "data" / "original").is_dir():
            return current

        if current == current.parent:
            break

        current = current.parent

    raise FileNotFoundError(
        "Could not locate the project root containing data/original/."
    )

In [4]:
PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
ORIGINAL_DIR = DATA_DIR / "original"
PUNS_PATH = DATA_DIR / "puns.json"
RESULTS_DIR = PROJECT_ROOT / "results" / "bertimbau"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

if not PUNS_PATH.is_file():
    raise FileNotFoundError(
        f"Missing file: {PUNS_PATH}"
    )

print("Project root:", PROJECT_ROOT)
print("Original corpus:", ORIGINAL_DIR)
print("Puns:", PUNS_PATH)
print("Results:", RESULTS_DIR)

Project root: /home/avelar/pun-detection-split-analysis
Original corpus: /home/avelar/pun-detection-split-analysis/data/original
Puns: /home/avelar/pun-detection-split-analysis/data/puns.json
Results: /home/avelar/pun-detection-split-analysis/results/bertimbau


In [5]:
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn_version)
print("CUDA available:", torch.cuda.is_available())
print("CUDA:", torch.version.cuda)
print("Model seed:", MODEL_SEED)
print("Split seeds:", SPLIT_SEEDS)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("cuDNN:", torch.backends.cudnn.version())

Python: 3.14.4
Platform: Linux-7.0.0-28-generic-x86_64-with-glibc2.43
PyTorch: 2.13.0+cu130
Transformers: 5.15.0
NumPy: 2.5.2
Pandas: 3.0.5
scikit-learn: 1.9.0
CUDA available: True
CUDA: 13.0
Model seed: 40
Split seeds: [13, 21, 40, 42, 73]
GPU: NVIDIA GeForce RTX 3060
cuDNN: 92000


In [ ]:
def validate_transformers_api():
    parameters = set(
        inspect.signature(
            TrainingArguments.__init__
        ).parameters
    )

    required_parameters = {
        "output_dir",
        "num_train_epochs",
        "learning_rate",
        "per_device_train_batch_size",
        "per_device_eval_batch_size",
        "weight_decay",
        "warmup_ratio",
        "max_grad_norm",
        "logging_strategy",
        "eval_strategy",
        "save_strategy",
        "load_best_model_at_end",
        "metric_for_best_model",
        "greater_is_better",
        "save_total_limit",
        "report_to",
        "fp16",
        "seed",
        "data_seed",
    }

    missing_parameters = (
        required_parameters
        - parameters
    )

    if missing_parameters:
        raise RuntimeError(
            "The installed Transformers version does not expose the "
            "TrainingArguments API expected by this notebook. "
            f"Missing parameters: {sorted(missing_parameters)}"
        )

    print(
        "TrainingArguments API validated for Transformers",
        transformers.__version__,
    )

In [7]:
validate_transformers_api()

TrainingArguments API validated for Transformers 5.15.0


In [8]:
def set_model_seed():
    random.seed(
        MODEL_SEED
    )

    np.random.seed(
        MODEL_SEED
    )

    torch.manual_seed(
        MODEL_SEED
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed(
            MODEL_SEED
        )

        torch.cuda.manual_seed_all(
            MODEL_SEED
        )

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [9]:
def read_jsonl(file_path):
    rows = []

    with Path(file_path).open(
        "r",
        encoding="utf-8",
    ) as file:
        for line in file:
            line = line.strip()

            if line:
                rows.append(
                    json.loads(
                        line
                    )
                )

    return pd.DataFrame(
        rows
    )

In [10]:
def load_split_directory(split_dir):
    split_dir = Path(
        split_dir
    )

    paths = {
        split_name: (
            split_dir
            / f"{split_name}.jsonl"
        )
        for split_name in SPLIT_NAMES
    }

    for split_name, path in paths.items():
        if not path.is_file():
            raise FileNotFoundError(
                f"Missing {split_name} file: {path}"
            )

    return {
        split_name: read_jsonl(
            path
        )
        for split_name, path in paths.items()
    }

In [11]:
def validate_input_splits(
    split_data,
    run_name,
):
    all_ids = []

    for split_name in SPLIT_NAMES:
        split_df = split_data[
            split_name
        ]

        required_columns = {
            "id",
            "text",
            "label",
        }

        missing_columns = (
            required_columns
            - set(
                split_df.columns
            )
        )

        if missing_columns:
            raise ValueError(
                f"{run_name}/{split_name}: missing columns "
                f"{sorted(missing_columns)}."
            )

        expected_size = (
            EXPECTED_SPLIT_COUNTS[
                split_name
            ]
        )

        if len(split_df) != expected_size:
            raise ValueError(
                f"{run_name}/{split_name}: expected "
                f"{expected_size} examples, found {len(split_df)}."
            )

        split_df["label"] = (
            split_df["label"]
            .astype(int)
        )

        observed_classes = (
            split_df["label"]
            .value_counts()
            .sort_index()
            .to_dict()
        )

        expected_classes = (
            EXPECTED_CLASS_COUNTS[
                split_name
            ]
        )

        if observed_classes != expected_classes:
            raise ValueError(
                f"{run_name}/{split_name}: expected class distribution "
                f"{expected_classes}, found {observed_classes}."
            )

        if split_df[
            "id"
        ].duplicated().any():
            raise ValueError(
                f"{run_name}/{split_name}: duplicated IDs."
            )

        all_ids.extend(
            split_df[
                "id"
            ]
            .astype(str)
            .tolist()
        )

    if len(
        all_ids
    ) != 5700:
        raise ValueError(
            f"{run_name}: expected 5700 total IDs."
        )

    if len(
        set(
            all_ids
        )
    ) != 5700:
        raise ValueError(
            f"{run_name}: IDs overlap across splits."
        )

In [12]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print(
    "Tokenizer loaded:",
    MODEL_NAME,
)

Tokenizer loaded: neuralmind/bert-large-portuguese-cased


In [13]:
class PunDataset(Dataset):
    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length,
    ):
        self.texts = (
            dataframe[
                "text"
            ]
            .astype(str)
            .tolist()
        )

        self.labels = (
            dataframe[
                "label"
            ]
            .astype(int)
            .tolist()
        )

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(
        self
    ):
        return len(
            self.texts
        )

    def __getitem__(
        self,
        index,
    ):
        encoding = self.tokenizer(
            self.texts[
                index
            ],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {
            "input_ids": (
                encoding[
                    "input_ids"
                ]
                .squeeze(0)
            ),
            "attention_mask": (
                encoding[
                    "attention_mask"
                ]
                .squeeze(0)
            ),
            "labels": torch.tensor(
                self.labels[
                    index
                ],
                dtype=torch.long,
            ),
        }

        if "token_type_ids" in encoding:
            item[
                "token_type_ids"
            ] = (
                encoding[
                    "token_type_ids"
                ]
                .squeeze(0)
            )

        return item

In [14]:
def compute_metrics(
    eval_pred
):
    logits, labels = (
        eval_pred
    )

    predictions = np.argmax(
        logits,
        axis=1,
    )

    return {
        "accuracy": accuracy_score(
            labels,
            predictions,
        ),
        "precision_macro": precision_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "recall_macro": recall_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "f1_macro": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "precision_weighted": precision_score(
            labels,
            predictions,
            average="weighted",
            zero_division=0,
        ),
        "recall_weighted": recall_score(
            labels,
            predictions,
            average="weighted",
            zero_division=0,
        ),
        "f1_weighted": f1_score(
            labels,
            predictions,
            average="weighted",
            zero_division=0,
        ),
    }

In [ ]:
def evaluate_predictions(
    y_true,
    y_pred,
):
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=[
            0,
            1,
        ],
        target_names=[
            "non_pun",
            "pun",
        ],
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[
            0,
            1,
        ],
    )

    tn, fp, fn, tp = (
        cm.ravel()
    )

    metrics = {
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "precision_non_pun": float(
            report_dict[
                "non_pun"
            ][
                "precision"
            ]
        ),
        "recall_non_pun": float(
            report_dict[
                "non_pun"
            ][
                "recall"
            ]
        ),
        "f1_non_pun": float(
            report_dict[
                "non_pun"
            ][
                "f1-score"
            ]
        ),
        "precision_pun": float(
            report_dict[
                "pun"
            ][
                "precision"
            ]
        ),
        "recall_pun": float(
            report_dict[
                "pun"
            ][
                "recall"
            ]
        ),
        "f1_pun": float(
            report_dict[
                "pun"
            ][
                "f1-score"
            ]
        ),
        "precision_macro": float(
            precision_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "recall_macro": float(
            recall_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "f1_macro": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "precision_weighted": float(
            precision_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            )
        ),
        "recall_weighted": float(
            recall_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            )
        ),
        "f1_weighted": float(
            f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            )
        ),
        "tp": int(
            tp
        ),
        "tn": int(
            tn
        ),
        "fp": int(
            fp
        ),
        "fn": int(
            fn
        ),
    }

    return (
        metrics,
        report_dict,
        cm,
    )

## Anotações de trocadilhos


In [16]:
def load_pun_annotations(
    file_path
):
    with Path(
        file_path
    ).open(
        "r",
        encoding="utf-8",
    ) as file:
        data = json.load(
            file
        )

    if not isinstance(
        data,
        list,
    ):
        raise ValueError(
            "data/puns.json must contain a JSON list."
        )

    annotations = {}

    for item in data:
        pun_id = str(
            item[
                "id"
            ]
        )

        if pun_id in annotations:
            raise ValueError(
                f"Duplicated pun annotation ID: {pun_id}"
            )

        annotations[
            pun_id
        ] = item

    return annotations

In [17]:
puns_by_id = load_pun_annotations(
    PUNS_PATH
)

print(
    "Annotated puns:",
    len(
        puns_by_id
    ),
)

if len(
    puns_by_id
) != 2850:
    raise ValueError(
        f"Expected 2850 pun annotations, found {len(puns_by_id)}."
    )

Annotated puns: 2850


In [18]:
def corpus_id_to_pun_id(
    example_id,
):
    return re.sub(
        r"\.[HN]$",
        "",
        str(
            example_id
        ),
    )

In [19]:
def classify_punning_sign(
    sign
):
    homograph = bool(
        sign.get(
            "homograph",
            False,
        )
    )

    homophone = bool(
        sign.get(
            "homophone",
            False,
        )
    )

    if homograph and homophone:
        return "both"

    if homograph:
        return "homograph_only"

    if homophone:
        return "homophone_only"

    return "none"

In [20]:
def count_false_negative_punning_signs(
    test_df,
    y_pred,
):
    counts = {
        category: 0
        for category in ERROR_CATEGORIES
    }

    false_negative_count = 0
    annotated_false_negative_count = 0
    missing_annotation_count = 0

    test_ids = (
        test_df[
            "id"
        ]
        .astype(str)
        .tolist()
    )

    y_true = (
        test_df[
            "label"
        ]
        .astype(int)
        .to_numpy()
    )

    y_pred = np.asarray(
        y_pred
    ).astype(int)

    for (
        example_id,
        true_label,
        predicted_label,
    ) in zip(
        test_ids,
        y_true,
        y_pred,
    ):
        if not (
            true_label == 1
            and predicted_label == 0
        ):
            continue

        false_negative_count += 1

        pun_id = corpus_id_to_pun_id(
            example_id
        )

        annotation = puns_by_id.get(
            pun_id
        )

        if annotation is None:
            missing_annotation_count += 1
            continue

        annotated_false_negative_count += 1

        for sign in annotation.get(
            "signs",
            [],
        ):
            category = classify_punning_sign(
                sign
            )

            counts[
                category
            ] += 1

    return {
        "fn_instances": int(
            false_negative_count
        ),
        "annotated_fn_instances": int(
            annotated_false_negative_count
        ),
        "missing_fn_annotations": int(
            missing_annotation_count
        ),
        "none": int(
            counts[
                "none"
            ]
        ),
        "homophone_only": int(
            counts[
                "homophone_only"
            ]
        ),
        "homograph_only": int(
            counts[
                "homograph_only"
            ]
        ),
        "both": int(
            counts[
                "both"
            ]
        ),
        "total_punning_signs_in_fn": int(
            sum(
                counts.values()
            )
        ),
    }

In [21]:
def save_error_analysis_plot(
    error_analysis,
    output_path,
    title,
):
    labels = [
        "None",
        "Homophone only",
        "Homograph only",
        "Both",
    ]

    values = [
        error_analysis[
            "none"
        ],
        error_analysis[
            "homophone_only"
        ],
        error_analysis[
            "homograph_only"
        ],
        error_analysis[
            "both"
        ],
    ]

    fig, ax = plt.subplots(
        figsize=(
            7,
            4.5,
        )
    )

    bars = ax.bar(
        labels,
        values,
    )

    ax.set_ylabel(
        "Number of punning signs in false negatives"
    )

    ax.set_title(
        title
    )

    ax.tick_params(
        axis="x",
        rotation=15,
    )

    for bar, value in zip(
        bars,
        values,
    ):
        ax.text(
            (
                bar.get_x()
                + bar.get_width() / 2
            ),
            bar.get_height(),
            str(
                value
            ),
            ha="center",
            va="bottom",
        )

    fig.tight_layout()

    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(
        fig
    )

In [22]:
def load_fresh_model():
    previous_verbosity = (
        hf_logging.get_verbosity()
    )

    hf_logging.set_verbosity_error()

    try:
        model = (
            AutoModelForSequenceClassification.from_pretrained(
                MODEL_NAME,
                num_labels=2,
                id2label=ID2LABEL,
                label2id=LABEL2ID,
            )
        )
    finally:
        hf_logging.set_verbosity(
            previous_verbosity
        )

    return model

In [ ]:
def build_training_arguments(
    checkpoint_dir
):
    return TrainingArguments(
        output_dir=str(
            checkpoint_dir
        ),
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="linear",
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        max_grad_norm=MAX_GRAD_NORM,
        logging_strategy="epoch",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=MODEL_SEED,
        data_seed=MODEL_SEED,
    )

In [ ]:
def build_run_metadata(
    condition,
    split_seed,
    split_dir,
    checkpoint_dir,
    trainer,
    train_df,
    validation_df,
    test_df,
):
    return {
        "model": MODEL_NAME,
        "condition": condition,
        "split_seed": split_seed,
        "model_seed": MODEL_SEED,
        "input_directory": str(
            Path(
                split_dir
            ).resolve()
        ),
        "checkpoint_directory": str(
            Path(
                checkpoint_dir
            ).resolve()
        ),
        "best_model_checkpoint": (
            trainer.state.best_model_checkpoint
        ),
        "train_examples": int(
            len(
                train_df
            )
        ),
        "validation_examples": int(
            len(
                validation_df
            )
        ),
        "test_examples": int(
            len(
                test_df
            )
        ),
        "max_length": MAX_LENGTH,
        "num_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "lr_scheduler_type": "linear",
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "max_grad_norm": MAX_GRAD_NORM,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "metric_for_best_model": "f1_macro",
        "fp16": bool(
            torch.cuda.is_available()
        ),
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn_version,
        "cuda_available": bool(
            torch.cuda.is_available()
        ),
        "cuda_version": torch.version.cuda,
        "gpu": (
            torch.cuda.get_device_name(
                0
            )
            if torch.cuda.is_available()
            else None
        ),
        "cudnn": (
            torch.backends.cudnn.version()
            if torch.cuda.is_available()
            else None
        ),
    }

In [25]:
def save_run_outputs(
    output_dir,
    metrics,
    report_dict,
    cm,
    error_analysis,
    metadata,
    training_history,
):
    output_dir = Path(
        output_dir
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    pd.DataFrame(
        [
            {
                "condition": metadata[
                    "condition"
                ],
                "split_seed": metadata[
                    "split_seed"
                ],
                "tp": metrics[
                    "tp"
                ],
                "tn": metrics[
                    "tn"
                ],
                "fp": metrics[
                    "fp"
                ],
                "fn": metrics[
                    "fn"
                ],
            }
        ]
    ).to_csv(
        output_dir
        / "confusion_counts.csv",
        index=False,
        encoding="utf-8",
    )

    pd.DataFrame(
        cm,
        index=[
            "true_non_pun",
            "true_pun",
        ],
        columns=[
            "pred_non_pun",
            "pred_pun",
        ],
    ).to_csv(
        output_dir
        / "confusion_matrix.csv",
        encoding="utf-8",
    )

    pd.DataFrame(
        [
            {
                "condition": metadata[
                    "condition"
                ],
                "split_seed": metadata[
                    "split_seed"
                ],
                **error_analysis,
            }
        ]
    ).to_csv(
        output_dir
        / "error_analysis.csv",
        index=False,
        encoding="utf-8",
    )

    pd.DataFrame(
        report_dict
    ).T.to_csv(
        output_dir
        / "classification_report.csv",
        encoding="utf-8",
    )

    pd.DataFrame(
        training_history
    ).to_csv(
        output_dir
        / "training_history.csv",
        index=False,
        encoding="utf-8",
    )

    with (
        output_dir
        / "metrics.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metrics,
            file,
            ensure_ascii=False,
            indent=2,
        )

    with (
        output_dir
        / "metadata.json"
    ).open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metadata,
            file,
            ensure_ascii=False,
            indent=2,
        )

    save_error_analysis_plot(
        error_analysis=error_analysis,
        output_path=(
            output_dir
            / "error_analysis.png"
        ),
        title=(
            metadata[
                "condition"
            ]
            if metadata[
                "split_seed"
            ] is None
            else (
                f"{metadata['condition']} "
                f"- seed {metadata['split_seed']}"
            )
        ),
    )

In [26]:
def prepare_run_directory(
    output_dir
):
    output_dir = Path(
        output_dir
    )

    checkpoint_dir = (
        output_dir
        / "checkpoints"
    )

    if checkpoint_dir.exists():
        shutil.rmtree(
            checkpoint_dir
        )

    checkpoint_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    return checkpoint_dir

In [27]:
def clear_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [28]:
def run_bertimbau(
    split_dir,
    condition,
    split_seed,
    output_dir,
):
    set_model_seed()

    split_data = load_split_directory(
        split_dir
    )

    validate_input_splits(
        split_data,
        condition,
    )

    train_df = split_data[
        "train"
    ]

    validation_df = split_data[
        "validation"
    ]

    test_df = split_data[
        "test"
    ]

    train_dataset = PunDataset(
        dataframe=train_df,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )

    validation_dataset = PunDataset(
        dataframe=validation_df,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )

    test_dataset = PunDataset(
        dataframe=test_df,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )

    output_dir = Path(
        output_dir
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    checkpoint_dir = prepare_run_directory(
        output_dir
    )

    set_model_seed()

    model = load_fresh_model()

    training_args = build_training_arguments(
        checkpoint_dir
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=EARLY_STOPPING_PATIENCE
            )
        ],
    )

    train_output = trainer.train()

    prediction_output = trainer.predict(
        test_dataset
    )

    y_true = (
        prediction_output.label_ids
    )

    y_pred = np.argmax(
        prediction_output.predictions,
        axis=1,
    )

    (
        metrics,
        report_dict,
        cm,
    ) = evaluate_predictions(
        y_true,
        y_pred,
    )

    error_analysis = (
        count_false_negative_punning_signs(
            test_df=test_df,
            y_pred=y_pred,
        )
    )

    prediction_metrics = (
        prediction_output.metrics
    )

    if (
        "test_loss"
        in prediction_metrics
    ):
        metrics[
            "test_loss"
        ] = float(
            prediction_metrics[
                "test_loss"
            ]
        )

    metrics[
        "best_validation_f1_macro"
    ] = (
        None
        if trainer.state.best_metric is None
        else float(
            trainer.state.best_metric
        )
    )

    metrics[
        "train_runtime"
    ] = float(
        train_output.metrics.get(
            "train_runtime",
            0.0,
        )
    )

    metadata = build_run_metadata(
        condition=condition,
        split_seed=split_seed,
        split_dir=split_dir,
        checkpoint_dir=checkpoint_dir,
        trainer=trainer,
        train_df=train_df,
        validation_df=validation_df,
        test_df=test_df,
    )

    save_run_outputs(
        output_dir=output_dir,
        metrics=metrics,
        report_dict=report_dict,
        cm=cm,
        error_analysis=error_analysis,
        metadata=metadata,
        training_history=trainer.state.log_history,
    )

    print("=" * 80)
    print("Condition:", condition)
    print("Split seed:", split_seed)

    print(
        classification_report(
            y_true,
            y_pred,
            labels=[
                0,
                1,
            ],
            target_names=[
                "non_pun",
                "pun",
            ],
            digits=4,
            zero_division=0,
        )
    )

    print("TP:", metrics["tp"])
    print("TN:", metrics["tn"])
    print("FP:", metrics["fp"])
    print("FN:", metrics["fn"])

    print(
        "Accuracy:",
        f"{metrics['accuracy']:.6f}",
    )

    print(
        "Macro-F1:",
        f"{metrics['f1_macro']:.6f}",
    )

    print(
        "Error analysis:",
        {
            category: error_analysis[
                category
            ]
            for category in ERROR_CATEGORIES
        },
    )

    result = {
        "condition": condition,
        "split_seed": split_seed,
        "model_seed": MODEL_SEED,
        **metrics,
        **error_analysis,
    }

    del prediction_output
    del train_dataset
    del validation_dataset
    del test_dataset
    del trainer
    del model

    clear_memory()

    return result

In [29]:
original_result = run_bertimbau(
    split_dir=ORIGINAL_DIR,
    condition="original",
    split_seed=None,
    output_dir=(
        RESULTS_DIR
        / "original"
    ),
)

original_result_df = pd.DataFrame(
    [
        original_result
    ]
)

display(
    original_result_df
)

original_result_df.to_csv(
    RESULTS_DIR
    / "original_result.csv",
    index=False,
    encoding="utf-8",
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
1,0.638222,0.625480,0.708772,0.723864,0.708772,0.703779,0.723864,0.708772,0.703779
2,0.503231,0.641663,0.687719,0.691117,0.687719,0.686325,0.691117,0.687719,0.686325
3,0.364883,1.044807,0.684211,0.690554,0.684211,0.681560,0.690554,0.684211,0.681560


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Condition: original
Split seed: None
              precision    recall  f1-score   support

     non_pun     0.7887    0.5632    0.6571       570
         pun     0.6603    0.8491    0.7429       570

    accuracy                         0.7061      1140
   macro avg     0.7245    0.7061    0.7000      1140
weighted avg     0.7245    0.7061    0.7000      1140

TP: 484
TN: 321
FP: 249
FN: 86
Accuracy: 0.706140
Macro-F1: 0.700007
Error analysis: {'none': 32, 'homophone_only': 17, 'homograph_only': 0, 'both': 39}


,condition,split_seed,model_seed,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,...,best_validation_f1_macro,train_runtime,fn_instances,annotated_fn_instances,missing_fn_annotations,none,homophone_only,homograph_only,both,total_punning_signs_in_fn
0,original,None,40,0.70614,0.724499,0.70614,0.700007,0.724499,0.70614,0.700007,...,0.703779,555.6506,86,86,0,32,17,0,39,88


In [30]:
pair_controlled_results = []

for split_seed in SPLIT_SEEDS:
    result = run_bertimbau(
        split_dir=(
            DATA_DIR
            / "pair_controlled"
            / f"seed_{split_seed}"
        ),
        condition="pair_controlled",
        split_seed=split_seed,
        output_dir=(
            RESULTS_DIR
            / "pair_controlled"
            / f"seed_{split_seed}"
        ),
    )

    pair_controlled_results.append(
        result
    )

pair_controlled_results_df = pd.DataFrame(
    pair_controlled_results
)

display(
    pair_controlled_results_df
)

pair_controlled_results_df.to_csv(
    RESULTS_DIR
    / "pair_controlled"
    / "runs.csv",
    index=False,
    encoding="utf-8",
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
1,0.657032,0.539682,0.756140,0.768438,0.756140,0.753315,0.768438,0.756140,0.753315
2,0.546194,0.474759,0.785965,0.787110,0.785965,0.785751,0.787110,0.785965,0.785751
3,0.425737,0.532779,0.807018,0.808247,0.807018,0.806825,0.808247,0.807018,0.806825
4,0.352055,0.718300,0.800000,0.804864,0.800000,0.799199,0.804864,0.800000,0.799199
5,0.273553,0.860162,0.794737,0.799790,0.794737,0.793868,0.799790,0.794737,0.793868


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Condition: pair_controlled
Split seed: 73
              precision    recall  f1-score   support

     non_pun     0.7192    0.8404    0.7751       570
         pun     0.8080    0.6719    0.7337       570

    accuracy                         0.7561      1140
   macro avg     0.7636    0.7561    0.7544      1140
weighted avg     0.7636    0.7561    0.7544      1140

TP: 383
TN: 479
FP: 91
FN: 187
Accuracy: 0.756140
Macro-F1: 0.754399
Error analysis: {'none': 81, 'homophone_only': 30, 'homograph_only': 0, 'both': 81}


,condition,split_seed,model_seed,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,...,best_validation_f1_macro,train_runtime,fn_instances,annotated_fn_instances,missing_fn_annotations,none,homophone_only,homograph_only,both,total_punning_signs_in_fn
0,pair_controlled,13,40,0.766667,0.767857,0.766667,0.766407,0.767857,0.766667,0.766407,...,0.806825,848.9207,152,152,0,71,25,1,55,152
1,pair_controlled,21,40,0.762281,0.764746,0.762281,0.761726,0.764746,0.762281,0.761726,...,0.779326,668.9665,163,163,0,66,30,3,67,166
2,pair_controlled,40,40,0.756140,0.757166,0.756140,0.755897,0.757166,0.756140,0.755897,...,0.745124,981.6578,157,157,0,72,22,2,65,161
3,pair_controlled,42,40,0.756140,0.756342,0.756140,0.756092,0.756342,0.756140,0.756092,...,0.785962,981.4870,147,147,0,69,27,2,52,150
4,pair_controlled,73,40,0.756140,0.763618,0.756140,0.754399,0.763618,0.756140,0.754399,...,0.772507,818.2819,187,187,0,81,30,0,81,192


In [31]:
max_cross_split_results = []

for split_seed in SPLIT_SEEDS:
    result = run_bertimbau(
        split_dir=(
            DATA_DIR
            / "max_cross_split"
            / f"seed_{split_seed}"
        ),
        condition="max_cross_split",
        split_seed=split_seed,
        output_dir=(
            RESULTS_DIR
            / "max_cross_split"
            / f"seed_{split_seed}"
        ),
    )

    max_cross_split_results.append(
        result
    )

max_cross_split_results_df = pd.DataFrame(
    max_cross_split_results
)

display(
    max_cross_split_results_df
)

max_cross_split_results_df.to_csv(
    RESULTS_DIR
    / "max_cross_split"
    / "runs.csv",
    index=False,
    encoding="utf-8",
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
1,0.644172,0.577187,0.691228,0.691568,0.691228,0.691091,0.691568,0.691228,0.691091
2,0.497066,0.716602,0.675439,0.692568,0.675439,0.668057,0.692568,0.675439,0.668057
3,0.360476,0.968196,0.663158,0.664136,0.663158,0.662655,0.664136,0.663158,0.662655


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Condition: max_cross_split
Split seed: 73
              precision    recall  f1-score   support

     non_pun     0.6446    0.6842    0.6638       570
         pun     0.6636    0.6228    0.6425       570

    accuracy                         0.6535      1140
   macro avg     0.6541    0.6535    0.6532      1140
weighted avg     0.6541    0.6535    0.6532      1140

TP: 355
TN: 390
FP: 180
FN: 215
Accuracy: 0.653509
Macro-F1: 0.653182
Error analysis: {'none': 107, 'homophone_only': 37, 'homograph_only': 0, 'both': 79}


,condition,split_seed,model_seed,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,...,best_validation_f1_macro,train_runtime,fn_instances,annotated_fn_instances,missing_fn_annotations,none,homophone_only,homograph_only,both,total_punning_signs_in_fn
0,max_cross_split,13,40,0.697368,0.697812,0.697368,0.697199,0.697812,0.697368,0.697199,...,0.691091,490.3007,186,186,0,97,38,0,53,188
1,max_cross_split,21,40,0.708772,0.715961,0.708772,0.706328,0.715961,0.708772,0.706328,...,0.729929,490.1429,218,218,0,102,40,0,82,224
2,max_cross_split,40,40,0.707018,0.707757,0.707018,0.706757,0.707757,0.707018,0.706757,...,0.721045,490.5349,150,150,0,62,25,0,66,153
3,max_cross_split,42,40,0.686842,0.692899,0.686842,0.684365,0.692899,0.686842,0.684365,...,0.679090,490.0005,229,229,0,98,38,1,97,234
4,max_cross_split,73,40,0.653509,0.654090,0.653509,0.653182,0.654090,0.653509,0.653182,...,0.651533,817.7172,215,215,0,107,37,0,79,223


In [32]:
review_results_df = pd.concat(
    [
        pair_controlled_results_df,
        max_cross_split_results_df,
    ],
    ignore_index=True,
)

all_results_df = pd.concat(
    [
        original_result_df,
        review_results_df,
    ],
    ignore_index=True,
)

display(
    all_results_df
)

all_results_df.to_csv(
    RESULTS_DIR
    / "all_runs.csv",
    index=False,
    encoding="utf-8",
)

,condition,split_seed,model_seed,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,...,best_validation_f1_macro,train_runtime,fn_instances,annotated_fn_instances,missing_fn_annotations,none,homophone_only,homograph_only,both,total_punning_signs_in_fn
0,original,None,40,0.706140,0.724499,0.706140,0.700007,0.724499,0.706140,0.700007,...,0.703779,555.6506,86,86,0,32,17,0,39,88
1,pair_controlled,13,40,0.766667,0.767857,0.766667,0.766407,0.767857,0.766667,0.766407,...,0.806825,848.9207,152,152,0,71,25,1,55,152
2,pair_controlled,21,40,0.762281,0.764746,0.762281,0.761726,0.764746,0.762281,0.761726,...,0.779326,668.9665,163,163,0,66,30,3,67,166
3,pair_controlled,40,40,0.756140,0.757166,0.756140,0.755897,0.757166,0.756140,0.755897,...,0.745124,981.6578,157,157,0,72,22,2,65,161
4,pair_controlled,42,40,0.756140,0.756342,0.756140,0.756092,0.756342,0.756140,0.756092,...,0.785962,981.4870,147,147,0,69,27,2,52,150
5,pair_controlled,73,40,0.756140,0.763618,0.756140,0.754399,0.763618,0.756140,0.754399,...,0.772507,818.2819,187,187,0,81,30,0,81,192
6,max_cross_split,13,40,0.697368,0.697812,0.697368,0.697199,0.697812,0.697368,0.697199,...,0.691091,490.3007,186,186,0,97,38,0,53,188
7,max_cross_split,21,40,0.708772,0.715961,0.708772,0.706328,0.715961,0.708772,0.706328,...,0.729929,490.1429,218,218,0,102,40,0,82,224
8,max_cross_split,40,40,0.707018,0.707757,0.707018,0.706757,0.707757,0.707018,0.706757,...,0.721045,490.5349,150,150,0,62,25,0,66,153
9,max_cross_split,42,40,0.686842,0.692899,0.686842,0.684365,0.692899,0.686842,0.684365,...,0.679090,490.0005,229,229,0,98,38,1,97,234


In [33]:
confusion_counts_all_df = all_results_df[
    [
        "condition",
        "split_seed",
        "model_seed",
        "tp",
        "tn",
        "fp",
        "fn",
    ]
].copy()

display(
    confusion_counts_all_df
)

confusion_counts_all_df.to_csv(
    RESULTS_DIR
    / "confusion_counts_all.csv",
    index=False,
    encoding="utf-8",
)

,condition,split_seed,model_seed,tp,tn,fp,fn
0,original,None,40,484,321,249,86
1,pair_controlled,13,40,418,456,114,152
2,pair_controlled,21,40,407,462,108,163
3,pair_controlled,40,40,413,449,121,157
4,pair_controlled,42,40,423,439,131,147
5,pair_controlled,73,40,383,479,91,187
6,max_cross_split,13,40,384,411,159,186
7,max_cross_split,21,40,352,456,114,218
8,max_cross_split,40,40,420,386,184,150
9,max_cross_split,42,40,341,442,128,229


In [34]:
error_analysis_all_df = all_results_df[
    [
        "condition",
        "split_seed",
        "model_seed",
        "fn_instances",
        "annotated_fn_instances",
        "missing_fn_annotations",
        "none",
        "homophone_only",
        "homograph_only",
        "both",
        "total_punning_signs_in_fn",
    ]
].copy()

display(
    error_analysis_all_df
)

error_analysis_all_df.to_csv(
    RESULTS_DIR
    / "error_analysis_all.csv",
    index=False,
    encoding="utf-8",
)

,condition,split_seed,model_seed,fn_instances,annotated_fn_instances,missing_fn_annotations,none,homophone_only,homograph_only,both,total_punning_signs_in_fn
0,original,None,40,86,86,0,32,17,0,39,88
1,pair_controlled,13,40,152,152,0,71,25,1,55,152
2,pair_controlled,21,40,163,163,0,66,30,3,67,166
3,pair_controlled,40,40,157,157,0,72,22,2,65,161
4,pair_controlled,42,40,147,147,0,69,27,2,52,150
5,pair_controlled,73,40,187,187,0,81,30,0,81,192
6,max_cross_split,13,40,186,186,0,97,38,0,53,188
7,max_cross_split,21,40,218,218,0,102,40,0,82,224
8,max_cross_split,40,40,150,150,0,62,25,0,66,153
9,max_cross_split,42,40,229,229,0,98,38,1,97,234


In [ ]:
SUMMARY_METRICS = [
    "accuracy",
    "precision_non_pun",
    "recall_non_pun",
    "f1_non_pun",
    "precision_pun",
    "recall_pun",
    "f1_pun",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "precision_weighted",
    "recall_weighted",
    "f1_weighted",
    "tp",
    "tn",
    "fp",
    "fn",
    "none",
    "homophone_only",
    "homograph_only",
    "both",
]

summary_rows = []

for strategy in STRATEGIES:
    strategy_df = (
        review_results_df.loc[
            review_results_df[
                "condition"
            ]
            == strategy
        ]
    )

    summary_row = {
        "condition": strategy,
        "runs": len(
            strategy_df
        ),
    }

    for metric in SUMMARY_METRICS:
        summary_row[
            f"{metric}_mean"
        ] = float(
            strategy_df[
                metric
            ].mean()
        )

        summary_row[
            f"{metric}_std"
        ] = float(
            strategy_df[
                metric
            ].std(
                ddof=1
            )
        )

    summary_rows.append(
        summary_row
    )

summary_df = pd.DataFrame(
    summary_rows
)

display(
    summary_df
)

summary_df.to_csv(
    RESULTS_DIR
    / "summary_mean_std.csv",
    index=False,
    encoding="utf-8",
)

,condition,runs,accuracy_mean,accuracy_std,precision_macro_mean,precision_macro_std,recall_macro_mean,recall_macro_std,f1_macro_mean,f1_macro_std,...,fn_mean,fn_std,none_mean,none_std,homophone_only_mean,homophone_only_std,homograph_only_mean,homograph_only_std,both_mean,both_std
0,pair_controlled,5,0.759474,0.004821,0.761946,0.004996,0.759474,0.004821,0.758904,0.005038,...,161.2,15.594871,71.8,5.630275,26.8,3.420526,1.6,1.140175,64.0,11.445523
1,max_cross_split,5,0.690702,0.022558,0.693704,0.023873,0.690702,0.022558,0.689566,0.022280,...,199.6,31.957785,93.2,17.880157,35.6,6.024948,0.2,0.447214,75.4,16.682326
